In [11]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score



In [13]:
# 1. Carregar a base limpa e aplicar a transformação binária
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    # Depressão (PHQ): itens, histórico, medicação, frequência, intensidade e severidade (target original)
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ',
    # Ansiedade (GAD-7): histórico, medicação, frequência, intensidade, itens e severidade
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]



In [14]:
# 2. Isolar X, y e Pesos nas Features Completas
y_bin = df[TARGET_BIN]
weights = df[PESO_COLUNA]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights[mascara_completos].copy()



In [15]:
# 3. Split Estratificado
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)



In [16]:
# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)



In [17]:
# 5. Configuração do Peso de Desbalanceamento para o XGBoost
scale_pos = (y_tr == 0).sum() / (y_tr == 1).sum()



In [18]:
# 6. Espaço de Busca de Hiperparâmetros do XGBoost
param_dist_xgb = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'scale_pos_weight': [scale_pos, scale_pos * 0.8, scale_pos * 1.2]
}



In [19]:
# 7. Execução do RandomizedSearchCV
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando otimização de hiperparâmetros do XGBoost...")
inicio = time.time()
random_search_xgb.fit(X_tr_proc, y_tr, sample_weight=w_tr)
tempo_total = time.time() - inicio

xgb_otimizado = random_search_xgb.best_estimator_

print(f"\n✓ Otimização concluída em {tempo_total:.2f}s")
print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (XGBOOST) ---")
for param, val in random_search_xgb.best_params_.items():
    print(f"{param}: {val}")



Iniciando otimização de hiperparâmetros do XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✓ Otimização concluída em 26.16s

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (XGBOOST) ---
subsample: 0.7
scale_pos_weight: 2.9652288965334748
n_estimators: 200
max_depth: 3
learning_rate: 0.1
gamma: 0.5
colsample_bytree: 0.8


In [20]:
# 8. Avaliação no Conjunto de Teste
y_pred_xgb = xgb_otimizado.predict(X_te_proc)
y_proba_xgb = xgb_otimizado.predict_proba(X_te_proc)[:, 1]

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO XGBOOST (PONDERADO) ---")
print(classification_report(y_te, y_pred_xgb, sample_weight=w_te, zero_division=0))

# Detalhamento dos Quadrantes da Matriz de Confusão
cm_amostra = confusion_matrix(y_te, y_pred_xgb)
tn, fp, fn, tp = cm_amostra.ravel()
total_amostra = len(y_te)

cm_pop = confusion_matrix(y_te, y_pred_xgb, sample_weight=w_te)
tn_w, fp_w, fn_w, tp_w = cm_pop.ravel()
total_pop = w_te.sum()

print("--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---")
print(f"Verdadeiros Negativos (TN): {tn:5d} | {tn/total_amostra*100:6.2f}% (Sem Depressão - Correto)")
print(f"Falsos Positivos      (FP): {fp:5d} | {fp/total_amostra*100:6.2f}% (Alarme Falso)")
print(f"Falsos Negativos      (FN): {fn:5d} | {fn/total_amostra*100:6.2f}% (Subdiagnóstico)")
print(f"Verdadeiros Positivos (TP): {tp:5d} | {tp/total_amostra*100:6.2f}% (Com Depressão - Capturado)")

print("\n--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---")
print(f"Verdadeiros Negativos (TN): {tn_w:11,.0f} | {tn_w/total_pop*100:6.2f}%")
print(f"Falsos Positivos      (FP): {fp_w:11,.0f} | {fp_w/total_pop*100:6.2f}%")
print(f"Falsos Negativos      (FN): {fn_w:11,.0f} | {fn_w/total_pop*100:6.2f}%")
print(f"Verdadeiros Positivos (TP): {tp_w:11,.0f} | {tp_w/total_pop*100:6.2f}%")

print(f"\nROC-AUC Score (XGBoost): {roc_auc_score(y_te, y_proba_xgb, sample_weight=w_te):.4f}")


--- RELATÓRIO DE CLASSIFICAÇÃO XGBOOST (PONDERADO) ---
              precision    recall  f1-score   support

           0       0.92      0.85      0.89 38513852.34799991
           1       0.56      0.73      0.63 9894699.189000001

    accuracy                           0.83 48408551.53699991
   macro avg       0.74      0.79      0.76 48408551.53699991
weighted avg       0.85      0.83      0.83 48408551.53699991

--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---
Verdadeiros Negativos (TN):  2929 |  65.86% (Sem Depressão - Correto)
Falsos Positivos      (FP):   573 |  12.89% (Alarme Falso)
Falsos Negativos      (FN):   246 |   5.53% (Subdiagnóstico)
Verdadeiros Positivos (TP):   699 |  15.72% (Com Depressão - Capturado)

--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---
Verdadeiros Negativos (TN):  32,763,990 |  67.68%
Falsos Positivos      (FP):   5,749,863 |  11.88%
Falsos Negativos      (FN):   2,713,776 |   5.61%
Verdadeiros Positivos (TP): 

### Análise Crítica dos Resultados: XGBoost (Gradient Boosting)

**Cenário avaliado:** previsão da depressão **sem sintomas de transtornos mentais em $X$**. Foram removidos os itens, o histórico, a medicação, a frequência, a intensidade e a severidade de depressão (PHQ) e de ansiedade (GAD-7), ou seja, as 25 colunas de `COLUNAS_SINTOMAS`. O modelo usa apenas variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

* **Capacidade Discriminativa (ROC-AUC = 0.8718):** Mesmo sem nenhum sintoma de saúde mental, o XGBoost mantém uma boa separação entre as classes. Na versão anterior, com as variáveis de ansiedade ainda em $X$, o ROC-AUC era 0.9092; a queda de cerca de 0.04 indica que parte do desempenho anterior vinha dessas variáveis (ansiedade e depressão são fortemente comórbidas), e não apenas do perfil sociodemográfico.
* **Sensibilidade Moderada (Recall = 72,6%):** Na projeção populacional, o modelo identificou corretamente **7.180.923 indivíduos com depressão** (14,83% da população), mas deixou de detectar **2.713.776** (falsos negativos = 5,61% da população, ou 27,4% dos casos reais).
* **Alarmes Falsos (Precisão = 55,5%):** Cerca de 55 em cada 100 pessoas classificadas como positivas têm depressão segundo o PHQ. Os falsos positivos somam **5.749.863 pessoas** (11,88% da população).
* **Equilíbrio Geral (F1-Score = 0.63; Acurácia = 82,5%):** A combinação de *Recall* de 72,6% com *Precisão* de 55,5% resulta em um F1-score de 0.63 para a classe positiva. O modelo acerta 82,5% das classificações em uma população na qual apenas 20,4% têm depressão (9.894.699 casos reais).
* **Conclusão Metodológica:** Com o mesmo split e as mesmas variáveis, o XGBoost supera ligeiramente o SVM em ROC-AUC (0.8718 vs. 0.8703), acurácia, precisão e F1, e gera cerca de 850 mil falsos positivos a menos; o SVM, por sua vez, é um pouco mais sensível (Recall = 75,4%) e deixa de detectar cerca de 275 mil casos a menos. As diferenças são pequenas e vêm de uma única partição de teste, de modo que os dois modelos devem ser vistos como equivalentes em poder discriminativo. O XGBoost é preferível quando a prioridade é **reduzir falsos alarmes**; o SVM, quando é **minimizar casos não detectados** de depressão.
* **Pendência (Random Forest):** O Random Forest (`03 Modelos supervisionados/Random Forest/06_02_preparacao_classificacao_2025.ipynb`) ainda foi treinado com as variáveis de ansiedade em $X$ (ROC-AUC = 0.9135; Recall = 71,76%). Esses valores **não são comparáveis** com os resultados acima, e o modelo precisa ser reexecutado com a mesma lista `COLUNAS_SINTOMAS` antes de qualquer comparação entre os três algoritmos.